# Adversarial Drift in Sequential Inference Systems
### A Hidden Failure Mode in Bayesian Estimators — Reproducible Analysis

This notebook reproduces the paper's core empirical result: a standard Kalman
filter silently fails under a **persistent (adversarial) additive measurement
bias**, while an **augmented, bias-aware Kalman filter** that jointly
estimates the bias as a latent state recovers substantially lower error.

All simulation logic lives in the `adrift` package under `src/`; this
notebook only orchestrates experiments and renders figures so that results
are reproducible from the command line via `scripts/run_experiment.py` as
well as interactively here.


In [ ]:
import os
import sys

sys.path.insert(0, os.path.join("..", "src"))

import numpy as np
import matplotlib.pyplot as plt

from adrift.config import SystemConfig
from adrift.metrics import format_report, summarize, percent_reduction
from adrift.simulate import run_multi_seed, run_single_seed
from adrift.plotting import plot_instant_error, plot_cumulative_error, plot_covariance_trace

RESULTS_DIR = os.path.join("..", "results")
os.makedirs(RESULTS_DIR, exist_ok=True)


## 1. System definition

True (misspecified) generative process:

$$x_{t+1} = A x_t + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$
$$y_t = C x_t + v_t + \delta, \quad v_t \sim \mathcal{N}(0, R)$$

where $\delta$ is a **persistent additive bias** that a standard Kalman
filter does not model. The proposed fix augments the state with a third
(near-constant) latent bias dimension so the filter can explain and correct
for it directly.


In [ ]:
cfg = SystemConfig(T=200, delta=0.5)

print("A =\n", cfg.A)
print("C =", cfg.C)
print("Q =\n", cfg.Q)
print("R =", cfg.R)
print()
print("A_aug =\n", cfg.A_aug)
print("C_aug =", cfg.C_aug)
print("Q_aug =\n", cfg.Q_aug)


## 2. Multi-seed cumulative-error experiment

We run the standard Kalman filter, the proposed bias-aware augmented Kalman
filter, and a naive (unfiltered) baseline across many random seeds, and
compare total (summed-over-time) estimation error.


In [ ]:
N_RUNS = 50
results = run_multi_seed(range(N_RUNS), cfg)
print(format_report(results))


## 3. Single-run visualization

For a representative single run, we plot instantaneous error, cumulative
error, and the standard Kalman filter's covariance trace (illustrating that
its reported confidence remains bounded even as its *actual* error grows
under the unmodeled bias — the core "hidden failure mode").


In [ ]:
single = run_single_seed(seed=0, cfg=cfg)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(single.err_kf, label="Kalman Filter")
axes[0].plot(single.err_bias_aware, label="Proposed (Bias-Aware)")
axes[0].set_title("Estimation Error under Persistent Bias")
axes[0].set_xlabel("Time Step")
axes[0].set_ylabel("Error (L2 Norm)")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(np.cumsum(single.err_kf), label="Kalman Filter")
axes[1].plot(np.cumsum(single.err_bias_aware), label="Proposed (Bias-Aware)")
axes[1].set_title("Cumulative Error under Persistent Bias")
axes[1].set_xlabel("Time Step")
axes[1].set_ylabel("Cumulative Error")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(single.kf_cov_trace)
axes[2].set_title("Kalman Filter Covariance Trace")
axes[2].set_xlabel("Time Step")
axes[2].set_ylabel("Trace(P)")
axes[2].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Persist individual figures (matching scripts/run_experiment.py output)
plot_instant_error(single.err_kf, single.err_bias_aware, RESULTS_DIR)
plot_cumulative_error(single.err_kf, single.err_bias_aware, RESULTS_DIR)
plot_covariance_trace(single.kf_cov_trace, RESULTS_DIR)
print(f"Figures written to {RESULTS_DIR}/")


## 4. Sensitivity to bias magnitude

We repeat the multi-seed experiment for a sweep of bias magnitudes $\delta$
to show that the proposed filter's advantage grows with the severity of the
adversarial drift, while a well-specified ($\delta = 0$) system shows no
meaningful difference between the two filters.


In [ ]:
deltas = [0.0, 0.1, 0.25, 0.5, 1.0]
reductions = []

for d in deltas:
    cfg_d = SystemConfig(T=200, delta=d)
    res_d = run_multi_seed(range(30), cfg_d)
    reduction = percent_reduction(res_d["kalman_filter"], res_d["bias_aware_kalman_filter"])
    reductions.append(reduction)
    mean_kf, _ = summarize(res_d["kalman_filter"])
    mean_aware, _ = summarize(res_d["bias_aware_kalman_filter"])
    print(f"delta={d:>4}: KF mean={mean_kf:7.2f}  Bias-Aware mean={mean_aware:7.2f}  reduction={reduction:6.1f}%")

plt.figure(figsize=(6, 4))
plt.plot(deltas, reductions, marker="o")
plt.axhline(0, color="gray", linewidth=0.8)
plt.xlabel("Bias magnitude (delta)")
plt.ylabel("% error reduction vs. standard KF")
plt.title("Proposed Filter's Advantage Grows with Bias Severity")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "delta_sensitivity.png"))
plt.show()


## Summary

- A standard Kalman filter is **misspecified** under persistent additive
  measurement bias: the bias leaks directly into the state estimate every
  step, and the filter's reported covariance does *not* reflect this
  degraded accuracy — the failure is silent.
- Augmenting the state with a latent (near-constant) bias term and jointly
  estimating it restores substantially lower cumulative error, with the
  improvement scaling with bias severity.
- All results here are exactly reproducible via
  `python scripts/run_experiment.py` from the repository root.
